# Clasificador de Sonidos: voz / sin voz / silencio

Árbol de decisión entrenado con **energía** y **periodicidad**.

## 1. Carga del dataset

In [ ]:
import pandas as pd

csv_path = "dataset_energia_periodicidad.csv"
df = pd.read_csv(csv_path)

energy_candidates     = ["energia", "energía", "Energy", "energy"]
periodicity_candidates = ["periodicidad", "periodicity", "Periodicidad"]
target_candidates     = ["etiqueta", "[etiqueta]", "label", "target"]

def find_column(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    raise ValueError(f"No se encontró ninguna de: {candidates}. Disponibles: {list(df.columns)}")

energy_col      = find_column(df, energy_candidates)
periodicity_col = find_column(df, periodicity_candidates)
target_col      = find_column(df, target_candidates)

df_model = df[[energy_col, periodicity_col, target_col]].dropna().copy()
X = df_model[[energy_col, periodicity_col]]
y = df_model[target_col]

print(f"Energía:      {energy_col}")
print(f"Periodicidad: {periodicity_col}")
print(f"Target:       {target_col}")
print(f"Shape:        {df_model.shape}")
display(df_model.head())


## 2. Distribución de clases

Antes de entrenar, conviene verificar que las clases estén balanceadas.

In [ ]:
import matplotlib.pyplot as plt

counts = y.value_counts()
colores = ["#4C72B0", "#DD8452", "#55A868"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(counts.index, counts.values, color=colores[:len(counts)], edgecolor="white", linewidth=0.8)
ax.bar_label(bars, fmt="%d", padding=4, fontsize=11)
ax.set_title("Distribución de clases", fontsize=14, fontweight="bold")
ax.set_xlabel("Clase")
ax.set_ylabel("Cantidad de muestras")
ax.set_ylim(0, counts.max() * 1.15)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print(counts.to_string())


## 3. Separación de datos y entrenamiento

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tree_model = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)
tree_model.fit(X_train, y_train)

print(f"Train: {X_train.shape[0]} muestras | Test: {X_test.shape[0]} muestras")
print("Modelo entrenado correctamente.")


## 4. Evaluación básica

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = tree_model.predict(X_test)

acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))


## 5. Matriz de confusión

Permite identificar **qué clases se confunden entre sí**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# — izquierda: conteos absolutos —
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de confusión\n(conteos)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Predicción")
axes[0].set_ylabel("Real")

# — derecha: normalizada por fila (recall por clase) —
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
im = axes[1].imshow(cm_norm, interpolation="nearest", cmap="RdYlGn", vmin=0, vmax=1)
plt.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_xticks(range(len(labels))); axes[1].set_xticklabels(labels)
axes[1].set_yticks(range(len(labels))); axes[1].set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        color = "white" if cm_norm[i, j] > 0.6 else "black"
        axes[1].text(j, i, f"{cm_norm[i,j]:.2f}", ha="center", va="center",
                     fontsize=12, fontweight="bold", color=color)
axes[1].set_title("Matriz de confusión\n(normalizada por clase)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Predicción")
axes[1].set_ylabel("Real")

plt.suptitle("Clasificador voz / sin voz / silencio", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("confusion_matrix_sound.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardada como confusion_matrix_sound.png")


## 6. Métricas por clase (precision, recall, F1)

Visualizadas como barras para comparar clases de un vistazo.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(y_test, y_pred, labels=labels)

metricas = pd.DataFrame({
    "Clase":     labels,
    "Precision": precision,
    "Recall":    recall,
    "F1-score":  f1,
    "Soporte":   support.astype(int)
}).set_index("Clase")

display(metricas.round(3))

# Barras agrupadas
x = np.arange(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - width, precision, width, label="Precision", color="#4C72B0", edgecolor="white")
ax.bar(x,         recall,    width, label="Recall",    color="#55A868", edgecolor="white")
ax.bar(x + width, f1,        width, label="F1-score",  color="#DD8452", edgecolor="white")
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=12)
ax.set_ylim(0, 1.15)
ax.axhline(0.8, color="gray", linestyle="--", linewidth=0.8, label="Umbral 0.80")
ax.set_ylabel("Score"); ax.set_title("Métricas por clase", fontsize=14, fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("metricas_por_clase_sound.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Frontera de decisión

Muestra cómo el árbol divide el espacio energía-periodicidad.

In [ ]:
from matplotlib.colors import ListedColormap
import numpy as np

clases = tree_model.classes_
colores_fondo = ["#AEC6E8", "#FFCC99", "#A8D5A2"]
colores_punto = ["#1F4E79", "#A0522D", "#1A5C35"]
cmap_fondo = ListedColormap(colores_fondo[:len(clases)])

h = 0.005
x_min, x_max = X[energy_col].min() - 0.05,  X[energy_col].max() + 0.05
y_min, y_max = X[periodicity_col].min() - 0.05, X[periodicity_col].max() + 0.05
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = tree_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_num = np.array([list(clases).index(z) for z in Z]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z_num, alpha=0.35, cmap=cmap_fondo)
for i, clase in enumerate(clases):
    mask = y_test == clase
    ax.scatter(X_test.loc[mask, energy_col],
               X_test.loc[mask, periodicity_col],
               c=colores_punto[i % len(colores_punto)],
               label=clase, edgecolors="white", linewidths=0.5, s=60, alpha=0.9)
ax.set_xlabel(energy_col); ax.set_ylabel(periodicity_col)
ax.set_title("Frontera de decisión (conjunto de prueba)", fontsize=13, fontweight="bold")
ax.legend(); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.savefig("frontera_decision_sound.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Importancia de features

In [ ]:
importancias = pd.Series(tree_model.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(6, 3))
importancias.plot(kind="barh", ax=ax, color=["#4C72B0", "#DD8452"], edgecolor="white")
ax.set_title("Importancia de features (Gini)", fontsize=13, fontweight="bold")
ax.set_xlabel("Importancia relativa")
ax.spines[["top","right"]].set_visible(False)
for i, v in enumerate(importancias):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=11)
plt.tight_layout()
plt.savefig("importancia_features_sound.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Árbol de decisión y guardado del modelo

In [ ]:
from sklearn.tree import plot_tree
import joblib

plt.figure(figsize=(18, 8))
plot_tree(tree_model, feature_names=list(X.columns),
          class_names=[str(c) for c in tree_model.classes_],
          filled=True, rounded=True)
plt.title("Árbol de decisión — voz/sin voz/silencio", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("arbol_decision_sound.png", dpi=150, bbox_inches="tight")
plt.show()

joblib.dump(tree_model, "decision_tree_voice_unvoice_silence.pkl")
print("Modelo guardado como decision_tree_voice_unvoice_silence.pkl")
